In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("bat-fwf5j").project("newbat")
version = project.version(2)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 140.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to newbat-2 in yolov8:: 100%|██████████| 593/593 [00:00<00:00, 3486.40it/s]


In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.4 MB/s eta 0:00:00


In [ ]:
!yolo task=pose mode=train model=yolo11n-pose.pt data="/content/newbat-2/data.yaml" epochs=50 imgsz=1920


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/newbat-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1920, in

In [6]:
# [V20.2 최종형] 홈플레이트 평행 가이드라인 + 마진 0px + 경로 수정 완료
!pip install ultralytics
from google.colab import drive
drive.mount('/content/drive')

import cv2
import numpy as np
import os, time, unicodedata
from pathlib import Path
from ultralytics import YOLO

# --- [설정 및 경로] ---
DRIVE_ROOT = Path("/content/drive/MyDrive")
HOMEPLATE_MODEL_PATH = "/content/homeplate1.pt" # 업로드하신 로컬 경로
BAT_MODEL_PATH = "/content/best_3.pt"          # 업로드하신 로컬 경로
VIDEO_DIR = DRIVE_ROOT / "KBO_edit"
RESULT_VIDEOS_DIR = DRIVE_ROOT / "KBO_result_videos_v20_2"
os.makedirs(RESULT_VIDEOS_DIR, exist_ok=True)

def extract_ground_truth(filename):
    normalized_name = unicodedata.normalize('NFC', Path(filename).stem)
    if "노스윙" in normalized_name: return "NO SWING"
    elif "스윙" in normalized_name: return "SWING"
    return "UNKNOWN"

def detect_handedness(hp_keypoints):
    if len(hp_keypoints) < 4: return None
    pt0, pt1, back_point = hp_keypoints[0], hp_keypoints[1], hp_keypoints[3]
    if pt0[0] <= 1 or pt1[0] <= 1 or back_point[0] <= 1: return None
    front_avg_x = (pt0[0] + pt1[0]) / 2
    return back_point[0] > front_avg_x

def analyze_checkswing(video_path, hp_model, bat_model):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return "ERROR", {}

    vid_w, vid_h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(5)
    out = cv2.VideoWriter(str(RESULT_VIDEOS_DIR / f"{video_path.stem}_v20_2.mp4"), cv2.VideoWriter_fourcc(*'mp4v'), fps, (vid_w, vid_h))

    handedness_votes, hp_pts0, hp_pts1 = [], [], []
    temp_frames = []

    # 1단계: 홈플레이트 탐지 및 영상 캡처
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        temp_frames.append(frame)

        hp_res = hp_model(frame, conf=0.3, verbose=False)
        if hp_res[0].keypoints is not None and len(hp_res[0].keypoints) > 0:
            k = hp_res[0].keypoints[0].xy.cpu().numpy()[0]
            if len(k) >= 4 and k[0][0] > 1:
                hp_pts0.append(k[0])
                hp_pts1.append(k[1])
                hv = detect_handedness(k)
                if hv is not None: handedness_votes.append(hv)

    cap.release()
    if not hp_pts0:
        out.release(); return "UNDETERMINED", {}

    # --- [핵심] 홈플레이트 기준 보조선용 벡터 계산 ---
    hp_ref0 = np.median(hp_pts0, axis=0)
    hp_ref1 = np.median(hp_pts1, axis=0)
    dx = hp_ref1[0] - hp_ref0[0]
    dy = hp_ref1[1] - hp_ref0[1]
    mid_plate_x = (hp_ref0[0] + hp_ref1[0]) / 2
    is_left = sum(handedness_votes) > len(handedness_votes) / 2 if handedness_votes else True

    # 2단계: 배트 궤적 탐지
    SPATIAL_BUFFER = 250
    bat_trajectory = []
    for f_idx, frame in enumerate(temp_frames):
        bat_res = bat_model(frame, conf=0.1, verbose=False)
        if bat_res[0].keypoints is not None and len(bat_res[0].keypoints) > 0:
            if bat_res[0].boxes and len(bat_res[0].boxes) > 0:
                idx = bat_res[0].boxes.conf.argmax().item()
                bk = bat_res[0].keypoints[idx].xy.cpu().numpy()[0]
                bb = bat_res[0].boxes.xyxy[idx].cpu().numpy()
                if len(bk) >= 2 and bk[0][0] > 1 and bk[1][0] > 1:
                    if abs(((bb[0]+bb[2])/2) - mid_plate_x) <= SPATIAL_BUFFER:
                        blen = ((bk[0][0]-bk[1][0])**2 + (bk[0][1]-bk[1][1])**2)**0.5
                        if 100 <= blen <= 350:
                            bat_trajectory.append({'f_idx': f_idx, 'p0': bk[0], 'p1': bk[1], 'len': blen})

    if not bat_trajectory:
        out.release(); return "UNDETERMINED", {}

    # 3단계: 체크스윙 판정 로직 (마진 0px)
    CROSS_MARGIN = 0
    curr_cons, max_cons, judge_f, judge_len = 0, 0, -1, 0
    for b in bat_trajectory:
        crossed = False
        p0, p1 = b['p0'], b['p1']
        if p1[1] > p0[1]:
            if is_left: crossed = (p1[0] < p0[0] - CROSS_MARGIN)
            else: crossed = (p1[0] > p0[0] + CROSS_MARGIN)

        if crossed:
            curr_cons += 1
            if curr_cons >= 2 and judge_f == -1:
                judge_f, judge_len = b['f_idx'], b['len']
            if curr_cons > max_cons: max_cons = curr_cons
        else: curr_cons = 0

    swing_detected = max_cons >= 2
    if judge_f == -1: judge_f = bat_trajectory[len(bat_trajectory)//2]['f_idx']

    # 4단계: 입체적 시각화 (평행 보조선)
    traj_dict = {b['f_idx']: b for b in bat_trajectory}
    line_scale = 1500

    for f_idx, frame in enumerate(temp_frames):
        # 홈플레이트 앞 모서리 (노란색)
        cv2.line(frame, (int(hp_ref0[0]), int(hp_ref0[1])), (int(hp_ref1[0]), int(hp_ref1[1])), (0, 255, 255), 10)

        if f_idx in traj_dict:
            b = traj_dict[f_idx]
            # Knob(p0, 빨강) & Tip(p1, 초록) 평행 보조선
            for p, color in [(b['p0'], (0,0,255)), (b['p1'], (0,255,0))]:
                s_pt = (int(p[0] - line_scale*dx), int(p[1] - line_scale*dy))
                e_pt = (int(p[0] + line_scale*dx), int(p[1] + line_scale*dy))
                cv2.line(frame, s_pt, e_pt, color, 2)

            p0, p1 = b['p0'], b['p1']
            cv2.circle(frame, (int(p0[0]),int(p0[1])), 8, (0,0,255), -1)
            cv2.circle(frame, (int(p1[0]),int(p1[1])), 8, (0,255,0), -1)
            cv2.line(frame, (int(p0[0]),int(p0[1])), (int(p1[0]),int(p1[1])), (0,255,255), 3)

        if f_idx >= judge_f:
            res_txt = "SWING!" if swing_detected else "NO SWING"
            color = (0, 0, 255) if swing_detected else (0, 255, 0)
            cv2.putText(frame, res_txt, (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 3.0, color, 6)

        out.write(frame)
    out.release()
    return ("SWING" if swing_detected else "NO SWING"), {"judge_frame": judge_f, "judge_bat_len": round(judge_len, 1)}

def main():
    h_model = YOLO(HOMEPLATE_MODEL_PATH)
    b_model = YOLO(BAT_MODEL_PATH)
    video_files = sorted([f for f in VIDEO_DIR.iterdir() if f.suffix.lower() == ".mp4"])

    print("====================================")
    print("🔥 V20.2 KCI 최종 시스템 검증 시작 🔥")
    print("====================================")

    tp, tn, fp, fn = 0, 0, 0, 0
    for i, vf in enumerate(video_files):
        gt = extract_ground_truth(vf.name)
        pred, details = analyze_checkswing(vf, h_model, b_model)
        match = (pred == gt)
        print(f"[{i+1}/{len(video_files)}] {vf.name} | GT: {gt} | Pred: {pred} | {'✅' if match else '❌'}")
        if pred == "UNDETERMINED": fn += 1
        elif match:
            if gt == "SWING": tp += 1
            else: tn += 1
        else:
            if gt == "SWING": fn += 1
            else: fp += 1

    total = len(video_files)
    acc = (tp+tn)/total*100 if total > 0 else 0
    prec = tp/(tp+fp)*100 if (tp+fp) > 0 else 0
    rec = tp/(tp+fn)*100 if (tp+fn) > 0 else 0
    f1 = 2*prec*rec/(prec+rec)/100 if (prec+rec) > 0 else 0

    print(f"\n{'='*50}\n[최종 성능 지표]\n{'='*50}")
    print(f"Accuracy:  {acc:.2f}% | Precision: {prec:.2f}% | Recall: {rec:.2f}% | F1: {f1:.4f}")
    print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}\n{'='*50}")

if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔥 V20.2 KCI 최종 시스템 검증 시작 🔥
[1/145] 250819_SSGKT_8초_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | ✅
[2/145] 250820_SSGKT_8말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | ✅
[3/145] 250820_롯데LG_9초_노스윙.mp4 | GT: NO SWING | Pred: SWING | ❌
[4/145] 250820_삼성NC_9초_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | ✅
[5/145] 250820_키움KIA_7초_스윙.mp4 | GT: SWING | Pred: NO SWING | ❌
[6/145] 250821_SSGKT_5말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | ✅
[7/145] 250821_SSGKT_9말_스윙.mp4 | GT: SWING | Pred: SWING | ✅
[8/145] 250821_롯데LG_7말_스윙.mp4 | GT: SWING | Pred: SWING | ✅
[9/145] 250821_삼성NC_8초_스윙.mp4 | GT: SWING | Pred: SWING | ✅
[10/145] 250821_키움KIA_4말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | ✅
[11/145] 250822_KT두산_8초_스윙.mp4 | GT: SWING | Pred: NO SWING | ❌
[12/145] 250822_LGKIA_1말_노스윙.mp4 | GT: NO SWING